In [1]:
# ============================================================
# STEP 1: Load Datasets
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

nav_history = pd.read_csv("02_nav_history.csv")
benchmark = pd.read_csv("10_benchmark_indices.csv")

print("Datasets loaded successfully")

Datasets loaded successfully


In [2]:
# ============================================================
# STEP 2: Convert Date Columns
# ============================================================

nav_history["date"] = pd.to_datetime(nav_history["date"])
benchmark["date"] = pd.to_datetime(benchmark["date"])

print("Date conversion completed")

Date conversion completed


In [3]:
# ============================================================
# STEP 3: Daily Returns
# ============================================================

nav_history = nav_history.sort_values(
    ["amfi_code", "date"]
)

nav_history["daily_return"] = (
    nav_history
    .groupby("amfi_code")["nav"]
    .pct_change()
)

nav_history.head()

,amfi_code,date,nav,daily_return
5750,100016,2022-01-03,520.4608,NaN
5751,100016,2022-01-04,515.0971,-0.010306
5752,100016,2022-01-05,521.7239,0.012865
5753,100016,2022-01-06,515.7880,-0.011377
5754,100016,2022-01-07,515.1639,-0.001210


In [4]:
# ============================================================
# STEP 4: Average Daily Return
# ============================================================

avg_returns = (
    nav_history
    .groupby("amfi_code")["daily_return"]
    .mean()
)

avg_returns.head()

amfi_code
100016    0.000142
100025    0.000170
100033    0.001080
101206    0.000852
101207    0.000424
Name: daily_return, dtype: float64

In [5]:
# ============================================================
# STEP 5: Annualized Volatility
# ============================================================

volatility = (
    nav_history
    .groupby("amfi_code")["daily_return"]
    .std()
) * np.sqrt(252)

volatility.head()

amfi_code
100016    0.145481
100025    0.039052
100033    0.189367
101206    0.145682
101207    0.257973
Name: daily_return, dtype: float64

In [6]:
# ============================================================
# STEP 6: CAGR
# ============================================================

cagr_results = []

for fund in nav_history["amfi_code"].unique():

    temp = nav_history[
        nav_history["amfi_code"] == fund
    ]

    start_nav = temp["nav"].iloc[0]
    end_nav = temp["nav"].iloc[-1]

    years = (
        (temp["date"].max() -
         temp["date"].min()).days
    ) / 365

    cagr = (
        (end_nav / start_nav)
        ** (1 / years)
        - 1
    )

    cagr_results.append(
        [fund, cagr]
    )

cagr_df = pd.DataFrame(
    cagr_results,
    columns=["amfi_code", "cagr"]
)

cagr_df.head()

,amfi_code,cagr
0,100016,0.026352
1,100025,0.044551
2,100033,0.300997
3,101206,0.235205
4,101207,0.079331


In [7]:
risk_free_rate = 0.06

In [8]:
# ============================================================
# STEP 7: Sharpe Ratio
# ============================================================

sharpe_results = []

for fund in nav_history["amfi_code"].unique():

    temp = nav_history[
        nav_history["amfi_code"] == fund
    ]

    annual_return = (
        temp["daily_return"].mean()
        * 252
    )

    annual_volatility = (
        temp["daily_return"].std()
        * np.sqrt(252)
    )

    sharpe = (
        annual_return -
        risk_free_rate
    ) / annual_volatility

    sharpe_results.append(
        [fund, sharpe]
    )

sharpe_df = pd.DataFrame(
    sharpe_results,
    columns=["amfi_code", "sharpe_ratio"]
)

sharpe_df.head()

,amfi_code,sharpe_ratio
0,100016,-0.167148
1,100025,-0.439062
2,100033,1.120102
3,101206,1.061535
4,101207,0.182043


In [9]:
# ============================================================
# STEP 8: Maximum Drawdown
# ============================================================

drawdown_results = []

for fund in nav_history["amfi_code"].unique():

    temp = nav_history[
        nav_history["amfi_code"] == fund
    ].copy()

    temp["cum_max"] = (
        temp["nav"].cummax()
    )

    temp["drawdown"] = (
        temp["nav"] -
        temp["cum_max"]
    ) / temp["cum_max"]

    max_dd = (
        temp["drawdown"].min()
    )

    drawdown_results.append(
        [fund, max_dd]
    )

drawdown_df = pd.DataFrame(
    drawdown_results,
    columns=["amfi_code", "max_drawdown"]
)

drawdown_df.head()

,amfi_code,max_drawdown
0,100016,-0.247344
1,100025,-0.043083
2,100033,-0.162172
3,101206,-0.112916
4,101207,-0.354469


In [10]:
# ============================================================
# STEP 9: Performance Scorecard
# ============================================================

performance_metrics = pd.merge(
    cagr_df,
    sharpe_df,
    on="amfi_code"
)

performance_metrics = pd.merge(
    performance_metrics,
    drawdown_df,
    on="amfi_code"
)

performance_metrics.head()

,amfi_code,cagr,sharpe_ratio,max_drawdown
0,100016,0.026352,-0.167148,-0.247344
1,100025,0.044551,-0.439062,-0.043083
2,100033,0.300997,1.120102,-0.162172
3,101206,0.235205,1.061535,-0.112916
4,101207,0.079331,0.182043,-0.354469


In [11]:
# ============================================================
# STEP 10: Top Performing Funds
# ============================================================

top_funds = (
    performance_metrics
    .sort_values(
        "sharpe_ratio",
        ascending=False
    )
    .head(10)
)

top_funds

,amfi_code,cagr,sharpe_ratio,max_drawdown
27,120507,0.072324,1.508048,-0.000977
34,148567,0.309499,1.483518,-0.112657
30,120843,0.308833,1.338216,-0.129740
36,148569,0.319245,1.263220,-0.163967
19,119551,0.257849,1.244653,-0.150124
25,120505,0.328016,1.206020,-0.181885
38,149323,0.295581,1.160297,-0.172481
2,100033,0.300997,1.120102,-0.162172
9,118632,0.240312,1.116999,-0.174141
3,101206,0.235205,1.061535,-0.112916
